# Q3 · Step 4 — Automated machine learning with **TPOT**

TPOT searches for the best preprocessing + model + hyper-parameter combination with a genetic algorithm.
(Auto-sklearn only runs on Linux, so TPOT is used here because it also works on Windows/macOS.)

⏱ **Time budget:** `TPOT_MINUTES = 30` (the exam suggests 30–60 min). The notebook keeps working while TPOT searches — do not close it.
For a quick *dry-run* (to test everything works) set `TPOT_MINUTES = 2`, then set it back to 30+ for the final run.

> TPOT receives the **already-preprocessed numeric matrix** (same preprocessing as the baseline, fitted on the training set only), so the comparison with the baseline is fair.

In [1]:
# ---- Project setup: finds the project root automatically ----
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def find_root():
    start = Path(os.environ.get("LAB_ROOT", Path.cwd())).resolve()
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Project root not found. Open this notebook from inside the Prog_for_AI_Lab folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

Project root: C:\Users\hp\Desktop\Prog_for_AI_Lab\Prog_for_AI_Lab


In [2]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from lab_utils import save_json, load_json, savefig
import q3_common as q3
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

In [3]:
import tpot
from sklearn.pipeline import Pipeline
print("TPOT version:", tpot.__version__)

TPOT_MINUTES = int(os.environ.get("TPOT_MINUTES", 30))     # <-- change here (30-60 for the final run)
N_JOBS = min(4, os.cpu_count() or 1)
SEARCH_SPACE = "linear-light"      # fast, sklearn-only search space (good for 30 min on a laptop)
SEED = 42

df = q3.load_adult(ROOT / "data" / "q3_adult")
X_train, X_test, y_train, y_test = q3.split_data(df)
prep = q3.build_preprocessor().fit(X_train)
Xtr_p, Xte_p = prep.transform(X_train), prep.transform(X_test)
print("preprocessed train matrix:", Xtr_p.shape, "| time budget:", TPOT_MINUTES, "min | jobs:", N_JOBS)

TPOT version: 1.1.0
preprocessed train matrix: (26048, 106) | time budget: 30 min | jobs: 4


## Run the AutoML search

In [4]:
from tpot import TPOTClassifier

def run_tpot(n_jobs):
    est = TPOTClassifier(search_space=SEARCH_SPACE, scorers=["roc_auc"], scorers_weights=[1], cv=5,
                         max_time_mins=TPOT_MINUTES, max_eval_time_mins=5, n_jobs=n_jobs,
                         verbose=2, random_state=SEED)
    est.fit(Xtr_p, y_train.values)
    return est

t0 = time.perf_counter()
try:
    tpot_est = run_tpot(N_JOBS)
except Exception as e:                                   # parallel back-end problems (common on Windows) -> retry single-core
    print("Parallel run failed (", repr(e)[:200], ") -> retrying with n_jobs=1")
    N_JOBS = 1
    tpot_est = run_tpot(1)
search_time = time.perf_counter() - t0
best_pipeline = tpot_est.fitted_pipeline_
print(f"\nTPOT search finished in {search_time/60:.1f} minutes")

Generation: : 1it [32:48, 1968.10s/it]



TPOT search finished in 33.5 minutes


## Inspect the result

In [5]:
print("Best pipeline found by TPOT:\n"); print(best_pipeline)
automl_full = Pipeline([("prep", prep), ("automl", best_pipeline)])        # raw data -> prediction
m = q3.compute_metrics(automl_full, X_test, y_test)
print("\nTest metrics:", {k: round(v, 4) for k, v in m.items()})

Best pipeline found by TPOT:

Pipeline(steps=[('minmaxscaler', MinMaxScaler()),
                ('selectfwe', SelectFwe(alpha=0.0006973008821)),
                ('featureunion-1',
                 FeatureUnion(transformer_list=[('featureunion',
                                                 FeatureUnion(transformer_list=[('kbinsdiscretizer',
                                                                                 KBinsDiscretizer(encode='onehot-dense',
                                                                                                  n_bins=24,
                                                                                                  random_state=42,
                                                                                                  strategy=np.str_('uniform')))])),
                                                ('passthrough',
                                                 Passthrough())])),
                ('featureunion-2',
          

In [ ]:
joblib.dump(automl_full, ROOT / "models" / "automl_model.joblib")
save_json({"engine": "TPOT " + tpot.__version__, "search_space": SEARCH_SPACE, "time_budget_min": TPOT_MINUTES, "n_jobs": N_JOBS,
           "search_time_s": float(search_time), "best_pipeline": str(best_pipeline), "test_metrics": m,
           "cv_note": "internal 5-fold cross-validation on the training set, scorer = roc_auc"},
          ROOT / "results" / "q3" / "automl_results.json")
print("Saved models/automl_model.joblib and results/q3/automl_results.json ✔")

Saved models/automl_model.joblib and results/q3/automl_results.json ✔


: 